<div style="text-align:center; font-size:50px; font-weight:700; margin-top:25px;">
Python pour Data Science - Projet - Groupe 4A
</div>

<div style="text-align:center; font-size:40px; font-weight:700; margin-top:25px;">
Prédiction des prix immobiliers
</div>

<br>

<div style="display:flex; justify-content:space-between; margin-top:35px; font-size:20px;">

  <div style="width:45%;">
    <div style="font-size:22px; font-weight:700; margin-bottom:10px;">Étudiants :</div>
    <div>• Yassine MELLOUL</div>
    <div>• Amira BARHOUMI</div>
    <div>• Antoine FOUCART</div>
  </div>

  <div style="width:45%; text-align:right;">
    <div style="font-size:22px; font-weight:700; margin-bottom:10px;">Chargé de TD :</div>
    <div>Julien PRAMIL</div>
  </div>

</div>

<br>
<hr>

# **1 - DESCRIPTION ET PROBLEMATIQUE**

Dans un contexte où les prix immobiliers sont fortement hétérogènes selon les territoires, le marché du logement en France présente de fortes disparités liées à la localisation, à 

l’attractivité des zones et aux dynamiques socio-économiques locales. Cette variabilité rend complexe la compréhension des mécanismes de formation des prix. Ainsi, la problématique 

centrale de ce projet est d’identifier et quantifier les facteurs influençant le prix de l’immobilier à l’échelle des communes. Nous nous concentrons en particulier sur des déterminants 

socio-économiques tels que la densité de population, le taux de chômage, le revenu moyen et le taux de pauvreté, afin d’évaluer leur impact sur les niveaux de prix. Pour cela, nous 

mobilisons deux sources de données principales : la base DVF (Demandes de Valeurs Foncières), qui recense les transactions immobilières en France , et les données socio-économiques de 

l’INSEE . Les jeux de données utilisés sont accessibles respectivement sur data.gouv.fr et insee.fr, via les liens suivants : 
https://www.data.gouv.fr/datasets/demandes-de-valeurs-foncieres et https://www.insee.fr/fr/statistiques/5359146 .

In [ ]:
# IMPORTATION
# modules
import pandas as pd
import numpy as np

# fonctions
from src.data.collect import load__data_url_zip_txt, load_insee_dossier_complet
from src.data.clean import filter_dvf_columns, compute_prix_m2, preprocess_insee, merge_dvf_insee
from src.data.stats_desc import univariate_numeric_analysis, carte_dep_communes_cartiflette
#URL
url_dvf = "https://static.data.gouv.fr/resources/demandes-de-valeurs-foncieres/20260405-002321/valeursfoncieres-2025.txt.zip"
url_insee = "https://www.insee.fr/fr/statistiques/fichier/5359146/dossier_complet.zip"






# **2 - COLLECTE DE DONNEES ET NETTOYAGE**

## **2-1 COLLECTE**

In [ ]:
# collecte de la base dvf
dvf = load__data_url_zip_txt(url_dvf)

# aperçu des données brutes
summary_dvf = dvf.dtypes.to_frame(name="type")
summary_dvf["nb_valeurs_manquantes"] = dvf.isna().sum()

summary_dvf = summary_dvf.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = dvf.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_dvf)
print(dvf.head(2))

In [ ]:
# collecte de la base INSEE
insee = load_insee_dossier_complet(url_insee)

# aperçu des données brutes
summary_insee = insee.dtypes.to_frame(name="type")
summary_insee["nb_valeurs_manquantes"] = insee.isna().sum()

summary_insee = summary_insee.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = insee.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_insee)
print(insee.head(5))

## **2-1 NETTOYAGE**

### **Base DVF**
Le nettoyage des données DVF vise à construire un jeu de données cohérent et exploitable pour la modélisation du prix immobilier.

Dans un premier temps, seules les variables pertinentes sont conservées : prix de transaction, localisation (code postal, commune, département, code commune), caractéristiques du bien (type de local, surface bâtie) et type de mutation. Les variables inutiles ou trop détaillées sont supprimées.

Ensuite, plusieurs transformations sont appliquées :
- conversion du prix en format numérique (suppression des espaces, gestion des virgules),
- harmonisation des types (variables qualitatives en `string`),
- standardisation des codes géographiques (zfill pour obtenir des codes à 2 et 5 chiffres).

Un filtrage est réalisé pour ne conserver que les observations pertinentes :
- uniquement les ventes,
- prix et surface strictement positifs,
- biens de type *Maison* ou *Appartement*.

Les valeurs manquantes sont supprimées afin d’assurer la qualité des données.

Enfin, une variable dérivée **prix au m²** est calculée comme le ratio entre la valeur foncière et la surface bâtie.

Ce processus permet d’obtenir un jeu de données propre, homogène et directement utilisable pour l’analyse et la modélisation.



In [ ]:
# Nettoyage
dvf = filter_dvf_columns(dvf)
# Ajout de la varible prix du metre carré
dvf = compute_prix_m2(dvf)
# aperçu des données tratées
summary_dvf_f = dvf.dtypes.to_frame(name="type")
summary_dvf_f["nb_valeurs_manquantes"] = dvf.isna().sum()

summary_dvf_f = summary_dvf_f.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = dvf.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_dvf_f)
print(dvf.head(5))

### **Base INSEE**
La fonction `preprocess_insee` a pour objectif de transformer un DataFrame INSEE brut en un jeu de données propre, standardisé et exploitable pour l’analyse statistique ou la modélisation.

Dans un premier temps, une copie du DataFrame est créée afin d’éviter toute modification directe des données d’origine. Cette étape garantit une approche non destructive du traitement.

Ensuite, les colonnes sont renommées pour améliorer leur lisibilité et leur cohérence. Les noms techniques issus de l’INSEE (comme `CODGEO`, `MED21` ou `TP6021`) sont remplacés par des appellations explicites telles que `code_commune`, `mediane_niveau_vie` ou `taux_pauvrete`. Cette standardisation facilite l’écriture du code et la compréhension des variables.

La fonction procède ensuite à la conversion des types de données. Le code commune et la médiane du niveau de vie sont convertis en type `string`, car ils correspondent à des identifiants ou des valeurs textuelles. Le taux de pauvreté subit un nettoyage plus avancé : les valeurs non exploitables comme `"s"` ou `"nd"` sont remplacées par des valeurs manquantes (`NaN`), les virgules sont converties en points pour respecter le format numérique, puis la variable est transformée en nombre et divisée par 100 afin d’obtenir un taux compris entre 0 et 1.

À partir des variables existantes, de nouveaux indicateurs sont créés. Le taux de chômage est calculé en rapportant le nombre de chômeurs de 15 à 64 ans à la population totale, ce qui fournit une mesure simple de la pression du chômage. La densité de population est également construite en divisant la population par la superficie, permettant d’évaluer le degré de concentration des habitants sur le territoire.

Enfin, toutes les lignes contenant des valeurs manquantes sont supprimées avec `dropna()`, afin d’obtenir un jeu de données entièrement complet et cohérent pour les analyses futures.

In [ ]:
# Nettoyage
insee = preprocess_insee(insee)

# aperçu des données traitées
summary_insee_f = insee.dtypes.to_frame(name="type")
summary_insee_f["nb_valeurs_manquantes"] = insee.isna().sum()

summary_insee_f = summary_insee_f.reset_index().rename(columns={"index": "variable"})

n_rows, n_cols = insee.shape

print(f"Nombre de lignes : {n_rows}")
print(f"Nombre de colonnes : {n_cols}")

print(summary_insee_f)
print(insee.head(5))

### **Fusion des deux jeux de données selon code commune**

In [ ]:
# merge
data_final = merge_dvf_insee(dvf, insee)
# aperçu
print(data_final.head(6))


# **3 - ANALYSE DESCRIPTIVE**

## **3-1 - Univarié**

### **Valeur foncière**

In [ ]:
stats = univariate_numeric_analysis(dvf,"valeur_fonciere" )

### **prix du mètre carré**

In [ ]:
stats2 = univariate_numeric_analysis(dvf,"prix_m2" )

### densité de population

In [ ]:
stats3 = univariate_numeric_analysis(data_final,"densite" )

### superficie

In [ ]:
stats4 = univariate_numeric_analysis(data_final,"superficie" )

## **3-2 - Bivarié**

In [ ]:
carte_dep_communes_cartiflette(
    df=dvf,
    col="prix_m2",
    code_dep=35
)

# **4 - MODELISATION**